In [ ]:
%pip install ipython-sql
%reload_ext sql
%sql sqlite:///final.db

In [3]:
import prettytable
prettytable.DEFAULT = 'default'  # quick patch


In [34]:
sql="""DROP TABLE IF EXISTS BankAccounts;"""
%sql $sql

 * sqlite:///final.db
Done.


[]

In [35]:
sql="""CREATE TABLE BankAccounts (
    AccountNumber VARCHAR(5) NOT NULL,
    AccountName VARCHAR(25) NOT NULL,
    Balance DECIMAL(8,2) CHECK(Balance>=0) NOT NULL,
    PRIMARY KEY (AccountNumber)
    );"""
%sql $sql

 * sqlite:///final.db
Done.


[]

In [36]:
sql="""INSERT INTO BankAccounts VALUES
('B001','Rose',300),
('B002','James',10345),
('B003','Shoe Shop',124200),
('B004','Corner Shop',76000);"""
%sql $sql

 * sqlite:///final.db
4 rows affected.


[]

In [37]:
%sql SELECT * FROM BankAccounts


 * sqlite:///final.db
Done.


AccountNumber,AccountName,Balance
B001,Rose,300
B002,James,10345
B003,Shoe Shop,124200
B004,Corner Shop,76000


In [38]:
sql="""DROP TABLE IF EXISTS ShoeShop;"""
%sql $sql

 * sqlite:///final.db
Done.


[]

In [39]:
sql="""CREATE TABLE ShoeShop (
    Product VARCHAR(25) NOT NULL,
    Stock INTEGER NOT NULL,
    Price DECIMAL(8,2) CHECK(Price>0) NOT NULL,
    PRIMARY KEY (Product)
    )"""
%sql $sql

 * sqlite:///final.db
Done.


[]

In [40]:
sql="""INSERT INTO ShoeShop VALUES
('Boots',10,200),
('High heels',10,600),
('Brogues',10,150),
('Trainers',10,300);"""
%sql $sql

 * sqlite:///final.db
4 rows affected.


[]

In [41]:
%sql SELECT * FROM ShoeShop


 * sqlite:///final.db
Done.


Product,Stock,Price
Boots,10,200
High heels,10,600
Brogues,10,150
Trainers,10,300


In [19]:
%pip install mysql3

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mysql3 (from versions: none)
ERROR: No matching distribution found for mysql3


In [46]:
import sqlite3

# Connect to SQLite
con = sqlite3.connect("final.db")
cur = con.cursor()

try:
    # Start transaction
    cur.execute("BEGIN;")
    
    # Deduct 200 from Rose
    cur.execute("""
        UPDATE BankAccounts
        SET Balance = Balance - 1200
        WHERE AccountName = 'James';
    """)
    
    # Add 200 to Shoe Shop
    cur.execute("""
        UPDATE BankAccounts
        SET Balance = Balance + 1200
        WHERE AccountName = 'ShoeShop';
    """)
    
    # Reduce stock
    cur.execute("""
        UPDATE ShoeShop
        SET Stock = Stock - 4
        WHERE Product = 'Trainerhs';
    """)
    if cur.rowcount==0:
        raise Exception("product not found rolling back the transaction")
    
    # Deduct another 300 from Rose
    #cur.execute("""
    #    UPDATE BankAccounts
    #    SET Balance = Balance - 300
    #    WHERE AccountName = 'James';
    #""")

    # Commit changes
    con.commit()

except Exception as e:
    # Rollback if something goes wrong
    con.rollback()
    print("Transaction failed:", e)

finally:
    con.close()


Transaction failed: product not found rolling back the transaction


In [43]:
%sql SELECT * FROM BankAccounts;

 * sqlite:///final.db
Done.


AccountNumber,AccountName,Balance
B001,Rose,300
B002,James,9145
B003,Shoe Shop,124200
B004,Corner Shop,76000


In [44]:
%sql SELECT * FROM ShoeShop;

 * sqlite:///final.db
Done.


Product,Stock,Price
Boots,10,200
High heels,10,600
Brogues,10,150
Trainers,6,300
